# CFL quark-star deformation experiment

Edit the single settings cell and choose **Run All**. The first pass is a read-only preview with no scientific calculations or result writes. Review its case count, numerical stages, and destination. Change only `EXECUTE_REVIEWED_PLAN` to `True`, then **Run All** again in the same kernel to execute that exact plan once.

This notebook describes pure, cold, **bare self-bound CFL quark stars**, not hybrid stars or stars with a crust. It uses the approved full finite-strange-mass bag potential with the leading pairing term: $m_s=100$ MeV, $\Delta=100$ MeV, $B=57.5$ MeV fm$^{-3}$ (derived $B^{1/4}=144.97957215191494$ MeV), and $a_4=1$. These values, the finite-density surface, constants, and the frozen baseline hash are owned by the package and cannot be swept here. See [the scientific contract](../docs/cfl.md).

Stellar mode computes full sampled **M–R, Λ–M, and k₂–M sequences**, refined maximum masses when resolvable, and requested comparison masses. `FIXED_MASSES=[1.4]` adds a gravitational-mass comparison point; it does **not** restrict the sequence to 1.4 M☉. Keep at least one positive requested mass; an unbracketed target is reported as unavailable. Quick and strict use the same physical acceptance rules.

In [ ]:
from eos_generation.notebook import NotebookSettings, get_notebook_session

notebook_session = get_notebook_session()

## Experiment settings

`AMPLITUDES` is the dimensionless change in $c_s^2$ before the Gaussian and surface ramp are applied. `CENTER`, `WIDTH` (Gaussian σ), and `RAMP_WIDTH` use MeV fm$^{-3}$. The ramp starts at the fixed self-bound surface, ε = 190.2181760065314 MeV fm$^{-3}$, and reaches one after `RAMP_WIDTH`. The formula-derived upper endpoint is ε = 4008.81724402691 MeV fm$^{-3}$. There is no hadronic matching anchor or below-surface matter, and a causal failure anywhere in this complete domain rejects the proposal.

A scalar or list is accepted for each deformation control. Lists form a bounded Cartesian grid. The unchanged A=0 baseline is included automatically and shared between geometries. The checked-in **quick** example has seven amplitudes at one geometry and 119 sampled-sequence tidal targets before adaptive refinement. Change to **strict**, preview again, and explicitly execute for the governed convergence stages. Strict is more expensive; the preview shows all stages and target counts. Neither precision label guarantees physical validity or resolved observables.

In [ ]:
# Edit only this cell. Scalars or lists define the deformation grid.
AMPLITUDES = [-0.20, -0.10, 0.00, 0.10, 0.20,0.30,0.40]
CENTER = 250.0
WIDTH = 500.0
RAMP_WIDTH = 10.0

CALCULATION = "stellar"       # "stellar" or "thermodynamics"
FIXED_MASSES = [1.4]           # gravitational M/M_sun; extra points, not the sequence grid
PRECISION = "quick"           # "quick" or "strict"

EXECUTE_REVIEWED_PLAN = False  # preview first; True executes the unchanged reviewed plan

# Optional: reopen an existing experiment WITHOUT scientific calculations.
# Paste the completed experiment folder printed by a previous run.
LOAD_EXPERIMENT = None         # e.g. r"../runs/cfl_.../experiment_..."
BUILD_SAVED_PLOTS = False      # True only to build a missing view for LOAD_EXPERIMENT

In [ ]:
if not isinstance(EXECUTE_REVIEWED_PLAN, bool) or not isinstance(BUILD_SAVED_PLOTS, bool):
    raise TypeError("Execution and plotting controls must be exactly False or True.")
if LOAD_EXPERIMENT is not None and EXECUTE_REVIEWED_PLAN:
    raise ValueError("Choose load or execute, not both. Set EXECUTE_REVIEWED_PLAN=False to load.")
if BUILD_SAVED_PLOTS and LOAD_EXPERIMENT is None:
    raise ValueError("BUILD_SAVED_PLOTS is only for a saved LOAD_EXPERIMENT; new runs create their view automatically.")

settings = None
if LOAD_EXPERIMENT is None:
    settings = NotebookSettings.from_values(
        matter_model="cfl",
        epsilon_match="surface",
        amplitudes=AMPLITUDES,
        center=CENTER,
        width=WIDTH,
        ramp_width=RAMP_WIDTH,
        calculation=CALCULATION,
        fixed_masses=FIXED_MASSES,
        precision=PRECISION,
        diagnostics="off",  # extended CFL radial diagnostics are not established
    )

In [ ]:
notebook_run = None
experiment_result = None
if LOAD_EXPERIMENT is None:
    notebook_run = notebook_session.prepare(
        settings, record_preview=not EXECUTE_REVIEWED_PLAN
    )
    print(notebook_run.summary_text())
else:
    experiment_result = notebook_session.load(LOAD_EXPERIMENT)
    if experiment_result.settings.matter_model != "cfl":
        raise ValueError("LOAD_EXPERIMENT must refer to a CFL experiment.")
    print("Loaded and validated saved CFL results; 0 scientific solver calls.")

In [ ]:
if notebook_run is not None:
    experiment_result = notebook_session.execute(
        notebook_run,
        current_settings=settings,
        execute=EXECUTE_REVIEWED_PLAN,
    )
    if experiment_result is None:
        print("Preview only. Review the plan, then change EXECUTE_REVIEWED_PLAN to True and Run All again.")
    else:
        print(f"Completed experiment (copy this path to reopen): {experiment_result.experiment_path}")

In [ ]:
if experiment_result is not None:
    try:
        saved_view = notebook_session.present(
            experiment_result,
            create=EXECUTE_REVIEWED_PLAN or BUILD_SAVED_PLOTS,
        )
    except Exception:
        print("Saved scientific results are retained. If only presentation failed, reopen with LOAD_EXPERIMENT and BUILD_SAVED_PLOTS=True; do not rerun solvers to recover plots.")
        raise

## Reading your results

The combined view is a separate flat `plots/` folder beside the sealed experiment and `STUDENT_VIEW/`. It overlays accepted physical EoSs from every geometry, shows A=0 once, and contains M–R, Λ–M, k₂–M plus thermodynamic PNGs, combined thermodynamic/stellar CSV tables, a case catalogue, fixed-mass/maximum-mass tables, an availability inventory, and a hash manifest. `C000000` is the baseline and later six-digit C labels are convenient **local** labels, not persistent identifiers; canonical physical IDs and exact rejection reasons remain in the tables. Archive the entire timestamped run folder.

Rejected raw proposals receive no reconstruction or stellar calculations. A missing tidal curve is reported as unavailable, never invented. Curves use the final requested numerical stage; every stage, failure, surface-jump diagnostic, and A=0 identity check remains in the authoritative packets. No plot smooths or interpolates missing data, and failed sequence gaps are not bridged. The full sampled sequence may extend beyond the stable branch: curves alone are not a stability claim. A sampled peak is not a refined maximum mass. Maximum-mass availability and the configured mass-threshold comparison flag are separate from EoS validity and fixed-mass availability.

The finite self-bound surface tidal correction is applied exactly once by the solver, before k₂ and Λ are reported. Plotting does not apply it again. The deformed EoS is an effective cold one-fluid barotrope, not a microscopic proof of CFL composition throughout. Ordinary-nuclei/two-flavor stability is the explicitly retained external assumption in the frozen scientific contract.

To reopen results, set `EXECUTE_REVIEWED_PLAN=False`, paste the completed experiment path into `LOAD_EXPERIMENT`, and Run All. This is read-only by default. `BUILD_SAVED_PLOTS=True` explicitly creates a missing combined view from saved tables only; it never overwrites a view or reruns physics. For another scientific run, clear `LOAD_EXPERIMENT`, set `BUILD_SAVED_PLOTS=False`, and preview with execution disabled.